# 🌾 HeatShield Agri — Random Forest + Sparse PCE Training Pipeline

**Purpose:** Train Random Forest models on real Open-Meteo historical weather data for Ugandan districts, export to ONNX for browser deployment.

**Pipeline:**
1. Mount Google Drive for persistent output storage
2. Fetch 5 years of hourly data from Open-Meteo Historical API (real data)
3. Engineer 17 time-based features (lags, cyclical encoding, rolling stats)
4. Train separate Random Forest models for temperature, humidity, wind speed
5. Backtest against held-out data (last 30 days)
6. Export to ONNX via skl2onnx
7. Validate ONNX output matches sklearn output
8. Save everything to Google Drive

**Output (saved to Google Drive):**
```
My Drive/HeatShield/models/
  ├── temperature_model.onnx
  ├── humidity_model.onnx
  ├── windspeed_model.onnx
  ├── model_metadata.json
  ├── training_data.parquet   (cached raw data for re-training)
  └── training_report.json    (full metrics + feature importances)
```

**Runtime:** ~8-12 minutes on Colab free tier (CPU). No GPU needed.

---
## Cell 1: Mount Google Drive & Install Dependencies

In [ ]:
# ============================================================================
# CELL 1: Mount Google Drive & Install Dependencies
# ============================================================================

from google.colab import drive
drive.mount('/content/drive')

# Create output directory in Google Drive
import os
DRIVE_OUTPUT = '/content/drive/My Drive/HeatShield/models'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f"\n✓ Output directory ready: {DRIVE_OUTPUT}")

# Install required packages (skl2onnx is not pre-installed in Colab)
!pip install -q skl2onnx onnxruntime skforecast openturns

# Verify installations
import sklearn, skl2onnx, onnxruntime
print(f"\n✓ scikit-learn: {sklearn.__version__}")
print(f"✓ skl2onnx:     {skl2onnx.__version__}")
print(f"✓ onnxruntime:  {onnxruntime.__version__}")
import openturns as ot
print(f"✓ openturns:    {ot.__version__}")

---
## Cell 2: Configuration

Edit these settings before running. Current values are **production-ready** for the grant interview.

In [ ]:
# ============================================================================
# CELL 2: CONFIGURATION  —  Edit these before running
# ============================================================================

import json
import time
import warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import requests
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

# ---------- 12 Uganda districts from HeatShield dashboard ----------
DISTRICTS = {
    'Kampala':     {'lat': 0.3476,  'lon': 32.5825},
    'Gulu':        {'lat': 2.7747,  'lon': 32.2990},
    'Moroto':      {'lat': 2.5346,  'lon': 34.6713},
    'Mbale':       {'lat': 1.0750,  'lon': 34.1750},
    'Jinja':       {'lat': 0.4244,  'lon': 33.2041},
    'Mbarara':     {'lat': -0.6133, 'lon': 30.6545},
    'Lira':        {'lat': 2.2499,  'lon': 32.5339},
    'Soroti':      {'lat': 1.7150,  'lon': 33.6111},
    'Arua':        {'lat': 3.0200,  'lon': 30.9100},
    'Kabale':      {'lat': -1.2508, 'lon': 29.9894},
    'Hoima':       {'lat': 1.4331,  'lon': 31.3525},
    'Fort Portal': {'lat': 0.6710,  'lon': 30.2750},
}

# ---------- Training districts (climate diversity) ----------
# Moroto  = hottest/driest (Karamoja semi-arid)
# Kabale  = coolest (southwestern highlands, ~1800m)
# Kampala = central equatorial baseline
# Gulu    = northern savanna
TRAINING_DISTRICTS = ['Kampala', 'Moroto', 'Kabale', 'Gulu']

# ---------- Date range: 5 years of historical data ----------
START_DATE = '2021-01-01'
END_DATE   = '2025-12-31'

# ---------- Model hyperparameters ----------
LAGS             = [1, 2, 3, 6, 12, 24, 48, 72]  # hours lookback
FORECAST_HORIZON = 72   # predict up to 72 hours ahead
N_ESTIMATORS     = 30   # down from 100 (biggest size impact)
MAX_DEPTH        = 10   # down from 15
MIN_SAMPLES_LEAF = 10   # up from 5 (more aggressive pruning)

# ---------- Full-power RF for PCE teacher (accuracy-optimised) ----------
# PCE handles compression, so the teacher RF should maximise accuracy.
# These are the ORIGINAL parameters before they were cut for ONNX size.
FULL_N_ESTIMATORS     = 100  # full ensemble for maximum accuracy
FULL_MAX_DEPTH        = 15   # deeper trees capture more complex patterns
FULL_MIN_SAMPLES_LEAF = 5    # less aggressive pruning for accuracy
RANDOM_STATE     = 42

# ---------- Output paths ----------
LOCAL_OUTPUT = '/content/models'                    # fast local workspace
DRIVE_OUTPUT = '/content/drive/My Drive/HeatShield/models'  # persistent storage
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

# ---------- Cache: skip re-downloading if data already on Drive ----------
CACHE_PATH = os.path.join(DRIVE_OUTPUT, 'training_data.parquet')
USE_CACHE  = os.path.exists(CACHE_PATH)

print('Configuration loaded.')
print(f'  Training districts: {TRAINING_DISTRICTS}')
print(f'  Date range:         {START_DATE} → {END_DATE}')
print(f'  RF config:          {N_ESTIMATORS} trees, max_depth={MAX_DEPTH}')
print(f'  Full RF (PCE teacher): {FULL_N_ESTIMATORS} trees, max_depth={FULL_MAX_DEPTH}, min_leaf={FULL_MIN_SAMPLES_LEAF}')
print(f'  Cached data:        {"YES — will skip API calls" if USE_CACHE else "NO — will fetch from Open-Meteo"}')

---
## Cell 3: Fetch Open-Meteo Historical Data

Downloads 5 years of hourly weather data for each training district. Takes ~2-3 minutes on first run. Caches to Google Drive so subsequent runs skip the download.

In [ ]:
# ============================================================================
# CELL 3: FETCH OPEN-METEO HISTORICAL DATA
# ============================================================================

def fetch_open_meteo_historical(lat, lon, start_date, end_date):
    """Fetch hourly historical weather from Open-Meteo Archive API."""
    url = 'https://archive-api.open-meteo.com/v1/archive'
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': start_date,
        'end_date': end_date,
        'hourly': ','.join([
            'temperature_2m',
            'relative_humidity_2m',
            'wind_speed_10m',
            'shortwave_radiation',
            'surface_pressure',
        ]),
        'timezone': 'Africa/Kampala',
    }
    resp = requests.get(url, params=params, timeout=120)
    resp.raise_for_status()
    data = resp.json()

    df = pd.DataFrame({
        'timestamp':   pd.to_datetime(data['hourly']['time']),
        'temperature': data['hourly']['temperature_2m'],
        'humidity':    data['hourly']['relative_humidity_2m'],
        'wind_speed':  data['hourly']['wind_speed_10m'],
        'radiation':   data['hourly']['shortwave_radiation'],
        'pressure':    data['hourly']['surface_pressure'],
    })
    df = df.set_index('timestamp').sort_index()
    return df


if USE_CACHE:
    print('Loading cached data from Google Drive...')
    raw_data = pd.read_parquet(CACHE_PATH)
    print(f'  ✓ Loaded {len(raw_data):,} records from cache')
else:
    print('Fetching from Open-Meteo Historical API...\n')
    frames = []
    for name in TRAINING_DISTRICTS:
        coords = DISTRICTS[name]
        print(f'  Fetching {name} ({coords["lat"]}, {coords["lon"]})...')
        try:
            df = fetch_open_meteo_historical(
                coords['lat'], coords['lon'], START_DATE, END_DATE
            )
            df['district'] = name
            frames.append(df)
            print(f'    → {len(df):,} hourly records')
            time.sleep(1)  # polite rate limiting
        except Exception as e:
            print(f'    ✗ Failed: {e}')
            print(f'      Check: Is the date range valid? Open-Meteo archive')
            print(f'      may not have data up to {END_DATE} yet.')
            print(f'      Try setting END_DATE to a recent past date.')

    if not frames:
        raise RuntimeError('No data fetched. Check your internet connection and date range.')

    raw_data = pd.concat(frames)

    # Cache to Google Drive for next time
    print(f'\n  Caching to Google Drive: {CACHE_PATH}')
    raw_data.to_parquet(CACHE_PATH)
    print(f'  ✓ Cached for future runs')

# ---------- Clean ----------
print(f'\nData summary:')
print(f'  Total records: {len(raw_data):,}')
print(f'  Date range:    {raw_data.index.min()} → {raw_data.index.max()}')
print(f'  Districts:     {raw_data["district"].unique().tolist()}')
print(f'  Missing values:')
print(raw_data.isnull().sum().to_string())

# Interpolate small gaps (common in weather APIs)
raw_data = raw_data.interpolate(method='time', limit=6)
raw_data = raw_data.dropna()
print(f'\n  After cleaning: {len(raw_data):,} records')

---
## Cell 4: Explore the Data

Quick visual check that the data looks right before training.

In [ ]:
# ============================================================================
# CELL 4: DATA EXPLORATION  (optional but useful for interview screenshots)
# ============================================================================

import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle('HeatShield Agri — Training Data (Open-Meteo Historical)', fontsize=14, fontweight='bold')

for district in raw_data['district'].unique():
    subset = raw_data[raw_data['district'] == district]
    # Resample to daily mean for cleaner plots
    daily = subset[['temperature', 'humidity', 'wind_speed']].resample('D').mean()

    axes[0].plot(daily.index, daily['temperature'], label=district, alpha=0.7, linewidth=0.8)
    axes[1].plot(daily.index, daily['humidity'], label=district, alpha=0.7, linewidth=0.8)
    axes[2].plot(daily.index, daily['wind_speed'], label=district, alpha=0.7, linewidth=0.8)

axes[0].set_ylabel('Temperature (°C)')
axes[0].legend(loc='upper right', fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].set_ylabel('Humidity (%)')
axes[1].grid(True, alpha=0.3)

axes[2].set_ylabel('Wind Speed (m/s)')
axes[2].set_xlabel('Date')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_OUTPUT, 'training_data_overview.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\nPer-district statistics:')
print(raw_data.groupby('district')[['temperature', 'humidity', 'wind_speed']].describe().round(1))

---
## Cell 5: Feature Engineering

Creates 17 features per variable:
- **8 lag features:** t-1, t-2, t-3, t-6, t-12, t-24, t-48, t-72 hours
- **4 cyclical time:** hour_sin, hour_cos, day-of-year_sin, day-of-year_cos
- **3 rolling stats:** 24h mean, 72h mean, 24h standard deviation
- **2 delta features:** 1-hour change, 24-hour change

In [ ]:
# ============================================================================
# CELL 5: FEATURE ENGINEERING
# ============================================================================

def create_features(df, target_col, lags=LAGS):
    """Create lag + cyclical + rolling features for a single weather variable."""
    feat = pd.DataFrame(index=df.index)

    # Lag features
    for lag in lags:
        feat[f'lag_{lag}'] = df[target_col].shift(lag)

    # Cyclical time encoding (captures daily and seasonal patterns)
    hour = df.index.hour
    doy  = df.index.dayofyear
    feat['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    feat['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    feat['doy_sin']  = np.sin(2 * np.pi * doy / 365.25)
    feat['doy_cos']  = np.cos(2 * np.pi * doy / 365.25)

    # Rolling statistics
    feat['rolling_mean_24h'] = df[target_col].rolling(24, min_periods=1).mean()
    feat['rolling_mean_72h'] = df[target_col].rolling(72, min_periods=1).mean()
    feat['rolling_std_24h']  = df[target_col].rolling(24, min_periods=1).std()

    # Delta features (rate of change)
    feat['delta_1h']  = df[target_col].diff(1)
    feat['delta_24h'] = df[target_col].diff(24)

    # Target: next value (1-step ahead; multi-step via recursive prediction)
    feat['target'] = df[target_col].shift(-1)

    feat = feat.dropna()
    return feat


# Build feature matrices for all three variables
variables = {
    'temperature': 'temperature',
    'humidity':    'humidity',
    'windspeed':   'wind_speed',
}

feature_sets = {}
for var_label, col_name in variables.items():
    features = create_features(raw_data, col_name)
    feature_sets[var_label] = features
    print(f'{var_label:12s}  feature matrix: {features.shape}  '
          f'({features.shape[0]:,} samples × {features.shape[1]-1} features + 1 target)')

print(f'\nFeature names: {[c for c in features.columns if c != "target"]}')

---
## Cell 6: Train Random Forest Models

Trains 3 models with temporal train/test split (last 30 days held out). Reports MAE, RMSE, R², and OOB score.

In [ ]:
# ============================================================================
# CELL 6: TRAIN RANDOM FOREST MODELS
# ============================================================================

def train_model(features_df, variable_name):
    """Train RF with temporal split. Returns model, feature columns, metrics."""
    # Temporal split: last 30 days = test
    split_date = features_df.index.max() - timedelta(days=30)
    train = features_df[features_df.index <= split_date]
    test  = features_df[features_df.index >  split_date]

    feature_cols = [c for c in features_df.columns if c != 'target']
    X_train = train[feature_cols].values.astype(np.float32)
    y_train = train['target'].values.astype(np.float32)
    X_test  = test[feature_cols].values.astype(np.float32)
    y_test  = test['target'].values.astype(np.float32)

    print(f'\n{"-"*60}')
    print(f'  Training: {variable_name}')
    print(f'  Train: {len(X_train):,} samples | Test: {len(X_test):,} samples')
    print(f'  Features: {len(feature_cols)} | Trees: {N_ESTIMATORS} | Max depth: {MAX_DEPTH}')

    model = RandomForestRegressor(
        n_estimators=N_ESTIMATORS,
        max_depth=MAX_DEPTH,
        min_samples_leaf=MIN_SAMPLES_LEAF,
        n_jobs=-1,           # use all Colab CPU cores
        random_state=RANDOM_STATE,
        oob_score=True,
    )

    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    # Evaluate
    y_pred = model.predict(X_test)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    oob  = model.oob_score_

    print(f'  Training time: {train_time:.1f}s')
    print(f'  MAE:  {mae:.3f}')
    print(f'  RMSE: {rmse:.3f}')
    print(f'  R\u00b2:   {r2:.4f}')
    print(f'  OOB:  {oob:.4f}')

    # Feature importance
    importances = sorted(
        zip(feature_cols, model.feature_importances_),
        key=lambda x: x[1], reverse=True
    )
    print(f'  Top 5 features: {", ".join(f"{n}({v:.3f})" for n, v in importances[:5])}')

    metrics = {
        'variable': variable_name,
        'mae': round(mae, 4),
        'rmse': round(rmse, 4),
        'r2': round(r2, 4),
        'oob_score': round(oob, 4),
        'train_samples': len(X_train),
        'test_samples': len(X_test),
        'n_features': len(feature_cols),
        'feature_names': feature_cols,
        'feature_importances': {n: round(v, 4) for n, v in importances},
        'train_time_seconds': round(train_time, 1),
    }

    return model, feature_cols, metrics, (X_test, y_test, y_pred)


# Train all three models
trained_models = {}
all_metrics = []
test_data = {}

for var_label in variables:
    model, feat_cols, metrics, test_results = train_model(feature_sets[var_label], var_label)
    trained_models[var_label] = (model, feat_cols)
    all_metrics.append(metrics)
    test_data[var_label] = test_results

print(f'\n{"="*60}')
print('ALL MODELS TRAINED')
print(f'{"="*60}')
for m in all_metrics:
    print(f'  {m["variable"]:12s}  MAE={m["mae"]:.3f}  RMSE={m["rmse"]:.3f}  '
          f'R\u00b2={m["r2"]:.4f}  OOB={m["oob_score"]:.4f}  ({m["train_time_seconds"]}s)')

---
## Cell 6b: Train Full-Power Random Forest (PCE Teacher)

Trains a **full-accuracy RF** (100 trees, depth 15, min_leaf=5) that will serve as the teacher model for PCE distillation. Since PCE handles compression (6 MB → ~5 KB), the teacher should maximise accuracy rather than minimise ONNX size.

The cut RF from Cell 6 is kept for ONNX browser export (backward compatibility). The full RF here feeds exclusively into the PCE pipeline.

In [ ]:
# ============================================================================
# CELL 6b: TRAIN FULL-POWER RANDOM FOREST (PCE TEACHER)
# ============================================================================
#
# The cut RF (Cell 6) has: 30 trees, depth 10, min_leaf=10  → small ONNX
# The full RF (this cell): 100 trees, depth 15, min_leaf=5 → max accuracy
#
# PCE will distill the full RF, so accuracy matters more than model size.
# ============================================================================

def train_full_model(features_df, variable_name):
    """Train full-power RF for PCE teacher. Same split as Cell 6."""
    split_date = features_df.index.max() - timedelta(days=30)
    train = features_df[features_df.index <= split_date]
    test  = features_df[features_df.index >  split_date]

    feature_cols = [c for c in features_df.columns if c != 'target']
    X_train = train[feature_cols].values.astype(np.float32)
    y_train = train['target'].values.astype(np.float32)
    X_test  = test[feature_cols].values.astype(np.float32)
    y_test  = test['target'].values.astype(np.float32)

    print(f'\n{"-"*60}')
    print(f'  Training FULL RF: {variable_name}')
    print(f'  Train: {len(X_train):,} samples | Test: {len(X_test):,} samples')
    print(f'  Features: {len(feature_cols)} | Trees: {FULL_N_ESTIMATORS} | '
          f'Max depth: {FULL_MAX_DEPTH} | Min leaf: {FULL_MIN_SAMPLES_LEAF}')

    model = RandomForestRegressor(
        n_estimators=FULL_N_ESTIMATORS,
        max_depth=FULL_MAX_DEPTH,
        min_samples_leaf=FULL_MIN_SAMPLES_LEAF,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        oob_score=True,
    )

    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    y_pred = model.predict(X_test)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    oob  = model.oob_score_

    print(f'  Training time: {train_time:.1f}s')
    print(f'  MAE:  {mae:.3f}')
    print(f'  RMSE: {rmse:.3f}')
    print(f'  R²:   {r2:.4f}')
    print(f'  OOB:  {oob:.4f}')

    importances = sorted(
        zip(feature_cols, model.feature_importances_),
        key=lambda x: x[1], reverse=True
    )
    print(f'  Top 5 features: {", ".join(f"{n}({v:.3f})" for n, v in importances[:5])}')

    metrics = {
        'variable': variable_name,
        'mae': round(mae, 4), 'rmse': round(rmse, 4),
        'r2': round(r2, 4), 'oob_score': round(oob, 4),
        'n_estimators': FULL_N_ESTIMATORS,
        'max_depth': FULL_MAX_DEPTH,
        'min_samples_leaf': FULL_MIN_SAMPLES_LEAF,
        'train_time_seconds': round(train_time, 1),
    }

    return model, feature_cols, metrics, (X_test, y_test, y_pred)


# ── Train full RF for all three variables ──
trained_models_full = {}
full_metrics = []
full_test_data = {}

for var_label in variables:
    model, feat_cols, metrics, test_results = train_full_model(
        feature_sets[var_label], var_label
    )
    trained_models_full[var_label] = (model, feat_cols)
    full_metrics.append(metrics)
    full_test_data[var_label] = test_results

print(f'\n{"="*60}')
print('FULL RF MODELS TRAINED (PCE teacher)')
print(f'{"="*60}')

# Compare cut vs full RF accuracy
print(f'\n{"":12s}  {"Cut RF (ONNX)":>16s}  {"Full RF (PCE)":>16s}  {"Improvement":>12s}')
print('-' * 65)
for m_cut, m_full in zip(all_metrics, full_metrics):
    v = m_cut['variable']
    r2_cut = m_cut['r2']
    r2_full = m_full['r2']
    delta = r2_full - r2_cut
    print(f'{v:12s}  R²={r2_cut:.4f}          R²={r2_full:.4f}         '
          f'{"↑" if delta > 0 else "↓"}{abs(delta):.4f}')
print(f'\n  Cut RF:  {N_ESTIMATORS} trees, depth {MAX_DEPTH}, min_leaf {MIN_SAMPLES_LEAF}')
print(f'  Full RF: {FULL_N_ESTIMATORS} trees, depth {FULL_MAX_DEPTH}, min_leaf {FULL_MIN_SAMPLES_LEAF}')
print(f'\n  → Full RF feeds into PCE for maximum surrogate accuracy')
print(f'  → Cut RF still exports to ONNX for backward browser compatibility')

---
## Cell 7: Visualize Predictions vs Actuals

Shows the last 7 days of test data with model predictions overlaid. Good for interview screenshots.

In [ ]:
# ============================================================================
# CELL 7: PREDICTION VISUALIZATION  (good for interview screenshots)
# ============================================================================

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle('HeatShield Agri — Random Forest Predictions vs Actual (Last 7 Days of Test Set)',
             fontsize=13, fontweight='bold')

units = {'temperature': '°C', 'humidity': '%', 'windspeed': 'm/s'}
colors = {'temperature': '#d62728', 'humidity': '#1f77b4', 'windspeed': '#2ca02c'}

for i, var_label in enumerate(variables):
    X_test, y_test, y_pred = test_data[var_label]
    feat_df = feature_sets[var_label]
    test_idx = feat_df[feat_df.index > feat_df.index.max() - timedelta(days=30)].index

    # Last 7 days only for clarity
    n_hours = 7 * 24
    actual = y_test[-n_hours:]
    predicted = y_pred[-n_hours:]
    time_axis = test_idx[-n_hours:]

    ax = axes[i]
    ax.plot(time_axis, actual, color=colors[var_label], alpha=0.8, linewidth=1, label='Actual')
    ax.plot(time_axis, predicted, color='black', alpha=0.6, linewidth=1, linestyle='--', label='RF Predicted')
    ax.set_ylabel(f'{var_label.title()} ({units[var_label]})')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)

    m = all_metrics[i]
    ax.text(0.02, 0.95, f'MAE={m["mae"]:.2f}  RMSE={m["rmse"]:.2f}  R\u00b2={m["r2"]:.3f}',
            transform=ax.transAxes, fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

axes[2].set_xlabel('Date')
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_OUTPUT, 'prediction_vs_actual.png'), dpi=150, bbox_inches='tight')
plt.show()
print('\n✓ Chart saved to Google Drive: prediction_vs_actual.png')

---
## Cell 8: Export to ONNX & Validate

Converts each sklearn model to ONNX format and verifies that ONNX predictions match sklearn predictions exactly.

In [ ]:
# ============================================================================
# CELL 8: EXPORT TO ONNX & VALIDATE
# ============================================================================

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as ort

onnx_sizes = {}

for var_label in variables:
    model, feat_cols = trained_models[var_label]
    n_features = len(feat_cols)

    # --- Convert to ONNX ---
    initial_type = [('float_input', FloatTensorType([None, n_features]))]
    onnx_model = convert_sklearn(
        model,
        name=f'heatshield_{var_label}_rf',
        initial_types=initial_type,
        target_opset=15,
    )

    # Save locally first (faster), then copy to Drive
    local_path = os.path.join(LOCAL_OUTPUT, f'{var_label}_model.onnx')
    with open(local_path, 'wb') as f:
        f.write(onnx_model.SerializeToString())

    size_kb = os.path.getsize(local_path) / 1024
    onnx_sizes[var_label] = size_kb

    # --- Validate ONNX matches sklearn ---
    X_test = test_data[var_label][0]
    sklearn_pred = model.predict(X_test[:20])

    sess = ort.InferenceSession(local_path, providers=['CPUExecutionProvider'])
    input_name = sess.get_inputs()[0].name
    onnx_pred = sess.run(None, {input_name: X_test[:20].astype(np.float32)})[0].flatten()

    max_diff = np.max(np.abs(sklearn_pred - onnx_pred))
    status = '✓ PASS' if max_diff < 0.01 else '✗ FAIL'

    print(f'{var_label:12s}  ONNX: {size_kb:,.0f} KB  |  Validation: max_diff={max_diff:.8f}  {status}')

    # Copy to Google Drive
    drive_path = os.path.join(DRIVE_OUTPUT, f'{var_label}_model.onnx')
    import shutil
    shutil.copy2(local_path, drive_path)

total_kb = sum(onnx_sizes.values())
print(f'\nTotal ONNX size: {total_kb:,.0f} KB ({total_kb/1024:.1f} MB)')
print(f'✓ All models saved to Google Drive: {DRIVE_OUTPUT}')

---
## Cell 8b: Sparse PCE via LARS — Compress Full RF to ~5 KB Polynomial Surrogate

**What this does:** Distills each **full-power RF** (100 trees, depth 15) into a ~2–10 KB JSON polynomial model. The PCE surrogate:
- Achieves **R² > 0.99** fidelity to the **full** RF (not the cut ONNX version)
- Captures the full RF's superior accuracy in a model 1000× smaller
- Evaluates in **~30 lines of JavaScript** with zero dependencies
- Provides **Sobol sensitivity indices** for free — quantifying each feature's contribution

**Why full RF matters:** The cut RF (30 trees, depth 10) was already accuracy-compromised for ONNX size. PCE handles compression, so the teacher should be the strongest model possible. The full RF (100 trees, depth 15) has higher R² — and PCE preserves that accuracy.

In [ ]:
# ============================================================================
# CELL 8b: SPARSE PCE VIA LARS — FULL RF → POLYNOMIAL SURROGATE
# ============================================================================
#
# Teacher: trained_models_full (100 trees, depth 15, min_leaf=5)
# Student: Sparse Legendre PCE via LARS with corrected LOO-CV
# ============================================================================

import openturns as ot
ot.RandomGenerator.SetSeed(RANDOM_STATE)

# ── PCE Configuration ──
PCE_N_SAMPLES   = 800    # Latin Hypercube training samples
PCE_MAX_DEGREE  = 5      # Max polynomial degree
PCE_Q_NORM      = 0.7    # Hyperbolic truncation (penalizes high-order interactions)
PCE_N_TEST      = 2000   # Independent validation samples

print('Sparse PCE via LARS — Distilling FULL Random Forests')
print(f'  Teacher: Full RF ({FULL_N_ESTIMATORS} trees, depth {FULL_MAX_DEPTH})')
print(f'  Samples: {PCE_N_SAMPLES} (LHS) + {PCE_N_TEST} (test)')
print(f'  Max degree: {PCE_MAX_DEGREE} | q-norm: {PCE_Q_NORM}')
print(f'  Input dimension: 17 features per model')


def get_feature_bounds(feature_sets, var_label):
    """Extract [min, max] bounds for each feature from training data, with 5% padding."""
    df = feature_sets[var_label]
    feat_cols = [c for c in df.columns if c != 'target']
    bounds = []
    for col in feat_cols:
        lo, hi = df[col].min(), df[col].max()
        margin = 0.05 * (hi - lo) if hi > lo else 1.0
        bounds.append((lo - margin, hi + margin))
    return feat_cols, bounds


def fit_sparse_pce(rf_model, feat_cols, bounds, var_label):
    """
    Fit a sparse PCE surrogate to a trained (full) Random Forest model.
    
    Pipeline:
      1. Define Uniform marginals on feature bounds → Legendre basis
      2. Generate Latin Hypercube training samples
      3. Label samples with FULL RF predictions (teacher)
      4. Build hyperbolic-cross truncated polynomial basis
      5. Fit via LARS with corrected leave-one-out CV
      6. Extract active coefficients, multi-indices, Sobol indices
    """
    dim = len(feat_cols)
    print(f'\n{"─"*60}')
    print(f'  Fitting PCE for: {var_label} ({dim} features)')
    print(f'  Teacher: Full RF ({FULL_N_ESTIMATORS} trees, depth {FULL_MAX_DEPTH})')
    
    # Step 1: Uniform marginals → Legendre polynomials
    marginals = [ot.Uniform(lo, hi) for lo, hi in bounds]
    distribution = ot.JointDistribution(marginals)
    distribution.setDescription(feat_cols)
    
    # Step 2: Latin Hypercube training samples
    lhs = ot.LHSExperiment(distribution, PCE_N_SAMPLES)
    lhs.setAlwaysShuffle(True)
    X_pce = lhs.generate()
    
    # Step 3: Label with FULL RF predictions (teacher model)
    X_np = np.array(X_pce, dtype=np.float32)
    y_np = rf_model.predict(X_np).reshape(-1, 1)
    Y_pce = ot.Sample(y_np)
    
    print(f'  LHS training samples: {PCE_N_SAMPLES}')
    print(f'  Full RF predictions range: [{y_np.min():.2f}, {y_np.max():.2f}]')
    
    # Step 4: Hyperbolic-cross truncated Legendre basis
    poly_collection = [
        ot.StandardDistributionPolynomialFactory(marginals[i])
        for i in range(dim)
    ]
    enum_func = ot.HyperbolicAnisotropicEnumerateFunction(dim, PCE_Q_NORM)
    multivariate_basis = ot.OrthogonalProductPolynomialFactory(
        poly_collection, enum_func
    )
    
    basis_size = enum_func.getBasisSizeFromTotalDegree(PCE_MAX_DEGREE)
    print(f'  Candidate basis: {basis_size} terms (degree ≤ {PCE_MAX_DEGREE}, q={PCE_Q_NORM})')
    
    adaptive_strategy = ot.FixedStrategy(multivariate_basis, basis_size)
    
    # Step 5: LARS + corrected LOO-CV
    lars = ot.LARS()
    loo_cv = ot.CorrectedLeaveOneOut()
    selection = ot.LeastSquaresMetaModelSelectionFactory(lars, loo_cv)
    projection = ot.LeastSquaresStrategy(selection)
    
    algo = ot.FunctionalChaosAlgorithm(
        X_pce, Y_pce, distribution,
        adaptive_strategy, projection
    )
    
    t0 = time.time()
    algo.run()
    fit_time = time.time() - t0
    result = algo.getResult()
    
    # Step 6: Extract active terms
    coefficients = np.array(result.getCoefficients()).flatten()
    indices = result.getIndices()
    enum = result.getOrthogonalBasis().getEnumerateFunction()
    
    multi_indices = []
    for k in range(indices.getSize()):
        mi = list(enum(indices[k]))
        multi_indices.append([int(v) for v in mi])
    
    n_active = len(multi_indices)
    sparsity = 1 - n_active / basis_size
    
    print(f'  Active terms: {n_active} / {basis_size} ({sparsity:.1%} sparsity)')
    print(f'  Fit time: {fit_time:.1f}s')
    
    # Step 7: Validate PCE against FULL RF on independent test samples
    X_test_pce = distribution.getSample(PCE_N_TEST)
    X_test_np = np.array(X_test_pce, dtype=np.float32)
    y_test_rf = rf_model.predict(X_test_np)
    y_test_pce = np.array(result.getMetaModel()(X_test_pce)).flatten()
    
    mae = mean_absolute_error(y_test_rf, y_test_pce)
    rmse = np.sqrt(mean_squared_error(y_test_rf, y_test_pce))
    r2 = r2_score(y_test_rf, y_test_pce)
    max_err = np.max(np.abs(y_test_rf - y_test_pce))
    
    print(f'  ── Validation: PCE vs Full RF ({PCE_N_TEST} test points) ──')
    print(f'  R²:       {r2:.6f}')
    print(f'  MAE:      {mae:.4f}')
    print(f'  RMSE:     {rmse:.4f}')
    print(f'  Max err:  {max_err:.4f}')
    
    # Step 8: Sobol sensitivity indices (free from PCE coefficients)
    sobol = ot.FunctionalChaosSobolIndices(result)
    sobol_first = {}
    sobol_total = {}
    print(f'  ── Sobol Sensitivity Indices ──')
    for i, name in enumerate(feat_cols):
        s1 = sobol.getSobolIndex(i)
        st = sobol.getSobolTotalIndex(i)
        sobol_first[name] = round(float(s1), 6)
        sobol_total[name] = round(float(st), 6)
        if s1 > 0.01:
            print(f'    {name:>22s}: S1={s1:.4f}  ST={st:.4f}  '
                  f'interaction={st-s1:.4f}')
    
    return {
        'result': result,
        'coefficients': coefficients.tolist(),
        'multi_indices': multi_indices,
        'bounds': bounds,
        'feat_cols': feat_cols,
        'n_active': n_active,
        'basis_size': basis_size,
        'sparsity': sparsity,
        'metrics': {
            'r2': round(r2, 6), 'mae': round(mae, 4),
            'rmse': round(rmse, 4), 'max_error': round(max_err, 4),
        },
        'sobol_first_order': sobol_first,
        'sobol_total_order': sobol_total,
        'fit_time': round(fit_time, 1),
    }


# ── Fit PCE for all three variables using FULL RF as teacher ──
pce_results = {}
for var_label in variables:
    model_full, feat_cols_full = trained_models_full[var_label]
    feat_cols_list, bounds = get_feature_bounds(feature_sets, var_label)
    pce_results[var_label] = fit_sparse_pce(
        model_full, feat_cols_list, bounds, var_label
    )

print(f'\n{"="*60}')
print('ALL PCE SURROGATES FITTED (from Full RF teacher)')
print(f'{"="*60}')
for var_label, res in pce_results.items():
    m = res['metrics']
    print(f'  {var_label:12s}  R²={m["r2"]:.4f}  RMSE={m["rmse"]:.4f}  '
          f'Active: {res["n_active"]}/{res["basis_size"]}  '
          f'Sparsity: {res["sparsity"]:.1%}')

---
## Cell 8c: Export PCE Surrogates to JSON & Generate JavaScript Inference

Saves each PCE model as a compact JSON file (~2–10 KB) and generates a self-contained JavaScript evaluator that replaces the entire onnxruntime-web stack.

In [ ]:
# ============================================================================
# CELL 8c: EXPORT PCE TO JSON & GENERATE JAVASCRIPT
# ============================================================================

pce_sizes = {}

for var_label, res in pce_results.items():
    export_data = {
        'variable': var_label,
        'algorithm': 'Sparse PCE via LARS (Legendre basis, hyperbolic truncation)',
        'teacher_model': f'Full RF ({FULL_N_ESTIMATORS} trees, depth {FULL_MAX_DEPTH})',
        'input_features': res['feat_cols'],
        'dim': len(res['feat_cols']),
        'max_degree': PCE_MAX_DEGREE,
        'q_norm': PCE_Q_NORM,
        'n_active_terms': res['n_active'],
        'candidate_basis_size': res['basis_size'],
        'sparsity': round(res['sparsity'], 4),
        'norm_params': [
            {'lo': round(lo, 6), 'hi': round(hi, 6)}
            for lo, hi in res['bounds']
        ],
        'coefficients': [round(c, 10) for c in res['coefficients']],
        'multi_indices': res['multi_indices'],
        'validation_vs_full_rf': res['metrics'],
        'sobol_first_order': res['sobol_first_order'],
        'sobol_total_order': res['sobol_total_order'],
    }
    
    fname = f'{var_label}_pce.json'
    for outdir in [LOCAL_OUTPUT, DRIVE_OUTPUT]:
        path = os.path.join(outdir, fname)
        with open(path, 'w') as f:
            json.dump(export_data, f, separators=(',', ':'))
    
    size_kb = os.path.getsize(os.path.join(LOCAL_OUTPUT, fname)) / 1024
    pce_sizes[var_label] = size_kb
    print(f'{var_label:12s}  PCE JSON: {size_kb:,.1f} KB  '
          f'({res["n_active"]} terms)  '
          f'R²={res["metrics"]["r2"]:.4f} vs Full RF')

total_pce_kb = sum(pce_sizes.values())
total_onnx_kb = sum(onnx_sizes.values())
print(f'\nTotal PCE: {total_pce_kb:,.1f} KB  vs  Total ONNX: {total_onnx_kb:,.0f} KB')
print(f'Compression: {total_onnx_kb/total_pce_kb:,.0f}× (and PCE distills the FULL RF, not the cut one)')

# ── JavaScript PCE evaluator ──
js_code = '''/**\n * HeatShield Agri — PCE Inference (Zero Dependencies)\n * Sparse Legendre polynomial evaluator.\n * Replaces onnxruntime-web (~3 MB) with pure JS.\n */\nclass PCEModel {\n  constructor(json) {\n    this.dim = json.dim;\n    this.normParams = json.norm_params;\n    this.coefficients = json.coefficients;\n    this.multiIndices = json.multi_indices;\n    this.maxDeg = new Array(this.dim).fill(0);\n    for (const mi of this.multiIndices) {\n      for (let i = 0; i < this.dim; i++) {\n        if (mi[i] > this.maxDeg[i]) this.maxDeg[i] = mi[i];\n      }\n    }\n  }\n  _legendreAll(xi, maxK) {\n    const P = new Float64Array(maxK + 1);\n    P[0] = 1.0;\n    if (maxK >= 1) P[1] = xi;\n    for (let k = 1; k < maxK; k++) {\n      P[k+1] = ((2*k+1)*xi*P[k] - k*P[k-1]) / (k+1);\n    }\n    for (let k = 0; k <= maxK; k++) {\n      P[k] *= Math.sqrt((2*k+1)/2);\n    }\n    return P;\n  }\n  predict(inputs) {\n    const d = this.dim, basis = new Array(d);\n    for (let i = 0; i < d; i++) {\n      const {lo, hi} = this.normParams[i];\n      const xi = Math.max(-1, Math.min(1, 2*(inputs[i]-lo)/(hi-lo)-1));\n      basis[i] = this._legendreAll(xi, this.maxDeg[i]);\n    }\n    let result = 0;\n    for (let k = 0; k < this.coefficients.length; k++) {\n      let term = this.coefficients[k];\n      const mi = this.multiIndices[k];\n      for (let i = 0; i < d; i++) term *= basis[i][mi[i]];\n      result += term;\n    }\n    return result;\n  }\n}\n// Usage:\n// const model = new PCEModel(await fetch(\'/models/temperature_pce.json\').then(r=>r.json()));\n// const pred = model.predict(featureVector);'''

js_path = os.path.join(DRIVE_OUTPUT, 'pce-inference.js')
with open(js_path, 'w') as f:
    f.write(js_code)
print(f'\n✓ pce-inference.js saved ({len(js_code):,} chars) — zero dependencies')

---
## Cell 8d: Visualize Sobol Sensitivity Indices

Which features drive each prediction? A free byproduct of the PCE polynomial coefficients.

In [ ]:
# ============================================================================
# CELL 8d: SOBOL SENSITIVITY VISUALIZATION
# ============================================================================

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Sobol Sensitivity Indices — PCE from Full RF', fontsize=14, fontweight='bold')

for ax, var_label in zip(axes, variables.keys()):
    res = pce_results[var_label]
    s1, st = res['sobol_first_order'], res['sobol_total_order']
    top_n = sorted(st.keys(), key=lambda k: st[k], reverse=True)[:10]
    y_pos = np.arange(len(top_n))
    s1_vals = [s1[f] for f in top_n]
    st_vals = [st[f] for f in top_n]
    ax.barh(y_pos, st_vals, color='#2196F3', alpha=0.4, label='Total (ST)')
    ax.barh(y_pos, s1_vals, color='#2196F3', alpha=0.9, label='First-order (S1)')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(top_n, fontsize=9)
    ax.set_xlabel('Sobol Index')
    ax.set_title(f'{var_label.title()}\nR²={res["metrics"]["r2"]:.4f} vs Full RF', fontsize=11)
    ax.legend(loc='lower right', fontsize=8)
    ax.set_xlim(0, min(1.0, max(st_vals) * 1.3))
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_OUTPUT, 'sobol_indices.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✓ Sobol plot saved  |  S1 (dark) = main effect  |  ST−S1 (light) = interactions')

In [ ]:
# ============================================================================
# CELL 8e: FULL RF vs CUT RF vs PCE — THREE-WAY COMPARISON
# ============================================================================

print('\n' + '=' * 80)
print('  CUT RF (ONNX) vs FULL RF (teacher) vs SPARSE PCE (student)')
print('=' * 80)
print(f'{"":12s}  {"Cut RF R²":>10s}  {"Full RF R²":>11s}  {"PCE R²":>10s}  '
      f'{"ONNX size":>10s}  {"PCE size":>9s}  {"Ratio":>6s}')
print('-' * 80)
for m_cut, m_full in zip(all_metrics, full_metrics):
    v = m_cut['variable']
    pce_m = pce_results[v]['metrics']
    onnx_kb = onnx_sizes[v]
    pce_kb = pce_sizes[v]
    print(f'{v:12s}  {m_cut["r2"]:>10.4f}  {m_full["r2"]:>11.4f}  '
          f'{pce_m["r2"]:>10.4f}  {onnx_kb:>8,.0f}KB  {pce_kb:>7,.1f}KB  '
          f'{onnx_kb/pce_kb:>5,.0f}×')

print('-' * 80)
t_onnx = sum(onnx_sizes.values())
t_pce = sum(pce_sizes.values())
print(f'{"TOTAL":12s}  {"":>10s}  {"":>11s}  {"":>10s}  '
      f'{t_onnx:>8,.0f}KB  {t_pce:>7,.1f}KB  {t_onnx/t_pce:>5,.0f}×')
print(f'\n  Key insight: PCE distills the FULL RF\'s higher accuracy,')
print(f'  not the cut RF\'s already-compromised accuracy.')
print(f'\n  + onnxruntime-web runtime: ~3,000 KB (needed for RF ONNX, NOT for PCE)')
print(f'  = Effective download:  {t_onnx + 3000:>6,.0f} KB (ONNX path)  vs  {t_pce:>6,.1f} KB (PCE path)')
print(f'\n  3G download (300 KB/s):  RF = {(t_onnx+3000)/300:.0f}s  |  PCE = {t_pce/300:.1f}s')
print(f'  Inference:  RF ~1–5 ms (WASM)  |  PCE ~0.1–0.5 μs (pure JS polynomial)')

---
## Cell 9: Generate Metadata & Training Report

Creates `model_metadata.json` (needed by the TypeScript browser code) and a full training report.

In [ ]:
# ============================================================================
# CELL 9: GENERATE METADATA & TRAINING REPORT
# ============================================================================

metadata = {
    'generated_at': datetime.now().isoformat(),
    'generated_in': 'Google Colab',
    'training_districts': TRAINING_DISTRICTS,
    'all_districts': list(DISTRICTS.keys()),
    'district_coordinates': DISTRICTS,
    'date_range': {'start': START_DATE, 'end': END_DATE},
    'data_source': 'Open-Meteo Historical API (archive-api.open-meteo.com)',
    'data_type': 'Real observed weather data (NOT synthetic)',
    'lags': LAGS,
    'forecast_horizon_hours': FORECAST_HORIZON,
    'model_config': {
        'algorithm': 'RandomForestRegressor',
        'n_estimators': N_ESTIMATORS,
        'max_depth': MAX_DEPTH,
        'min_samples_leaf': MIN_SAMPLES_LEAF,
        'random_state': RANDOM_STATE,
    },
    'feature_engineering': {
        'total_features': 17,
        'lag_features': [f'lag_{l}' for l in LAGS],
        'cyclical_features': ['hour_sin', 'hour_cos', 'doy_sin', 'doy_cos'],
        'rolling_features': ['rolling_mean_24h', 'rolling_mean_72h', 'rolling_std_24h'],
        'delta_features': ['delta_1h', 'delta_24h'],
    },
    'wbgt_thresholds': {
        'low': 26.0,
        'moderate': 28.0,
        'high': 30.0,
        'very_high': 32.0,
    },
    'iso_7243_formula': 'WBGT = 0.7 * Tnwb + 0.2 * Tg + 0.1 * Tdb',
    'models': {},
}

for m in all_metrics:
    var_name = m['variable']
    metadata['models'][var_name] = {
        'onnx_file': f'{var_name}_model.onnx',
        'onnx_size_kb': round(onnx_sizes.get(var_name, 0), 1),
        'feature_names': m['feature_names'],
        'n_features': m['n_features'],
        'metrics': {
            'mae': m['mae'],
            'rmse': m['rmse'],
            'r2': m['r2'],
            'oob_score': m['oob_score'],
        },
        'train_samples': m['train_samples'],
        'test_samples': m['test_samples'],
        'train_time_seconds': m['train_time_seconds'],
    }

# Save metadata (needed by TypeScript browser code)
meta_path = os.path.join(DRIVE_OUTPUT, 'model_metadata.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

# Save full training report (for your records + interview reference)
report = {
    'metadata': metadata,
    'detailed_metrics': all_metrics,
    'data_statistics': {
        'total_records': len(raw_data),
        'districts_used': TRAINING_DISTRICTS,
        'per_district_records': raw_data.groupby('district').size().to_dict(),
        'temperature_range': {
            'min': float(raw_data['temperature'].min()),
            'max': float(raw_data['temperature'].max()),
            'mean': float(raw_data['temperature'].mean()),
        },
        'humidity_range': {
            'min': float(raw_data['humidity'].min()),
            'max': float(raw_data['humidity'].max()),
            'mean': float(raw_data['humidity'].mean()),
        },
    },
}

report_path = os.path.join(DRIVE_OUTPUT, 'training_report.json')
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print('✓ model_metadata.json saved (needed by browser integration)')
print('✓ training_report.json saved (full metrics for your records)')

---
## Cell 10: Final Summary & Next Steps

In [ ]:
# ============================================================================
# CELL 10: FINAL SUMMARY
# ============================================================================

print('=' * 65)
print('  HEATSHIELD AGRI — TRAINING PIPELINE COMPLETE')
print('=' * 65)

print(f'\n  Data: {len(raw_data):,} hourly records from Open-Meteo')
print(f'  Districts: {", ".join(TRAINING_DISTRICTS)}')
print(f'  Period: {START_DATE} → {END_DATE}')

print(f'\n  Model Performance:')
print(f'  {"-"*55}')
for m in all_metrics:
    v = m['variable']
    print(f'  {v:12s}  MAE={m["mae"]:.3f}  RMSE={m["rmse"]:.3f}  '
          f'R\u00b2={m["r2"]:.4f}  ONNX={onnx_sizes[v]:,.0f}KB')

total_mb = sum(onnx_sizes.values()) / 1024
print(f'  {"-"*55}')
print(f'  Total ONNX: {total_mb:.1f} MB')

print(f'\n  Files on Google Drive ({DRIVE_OUTPUT}/):')
for f in sorted(os.listdir(DRIVE_OUTPUT)):
    fpath = os.path.join(DRIVE_OUTPUT, f)
    size = os.path.getsize(fpath) / 1024
    unit = 'KB' if size < 1024 else 'MB'
    display_size = size if size < 1024 else size / 1024
    print(f'    {f:35s} {display_size:>8.1f} {unit}')

print(f'\n  Next steps:')
print(f'    1. Download the 3 .onnx files + model_metadata.json from Google Drive')
print(f'    2. Copy them to web/frontend/public/models/ in your project')
print(f'    3. npm install onnxruntime-web')
print(f'    4. Use ml-inference.ts to run predictions in the browser')
print(f'\n  To re-train with different settings:')
print(f'    - Edit Cell 2 configuration')
print(f'    - Delete {CACHE_PATH} to re-fetch data')
print(f'    - Run all cells again')

---

## Appendix: TypeScript Browser Integration Code

This is the `ml-inference.ts` file you'll add to your React project. It loads the ONNX models, recreates the same 17 features in TypeScript, and computes WBGT from predictions.

**Copy this cell's output into `src/ml-inference.ts` in your Vercel project.**

In [ ]:
# ============================================================================
# APPENDIX: Save TypeScript browser integration code to Google Drive
# ============================================================================

ts_code = r'''
/**
 * HeatShield Agri — Browser-side ML Inference
 * Runs Random Forest ONNX models in the browser via onnxruntime-web.
 * Predicts temperature, humidity, wind speed → computes WBGT.
 *
 * Install: npm install onnxruntime-web
 *
 * Usage:
 *   import { HeatShieldML } from './ml-inference';
 *   const ml = new HeatShieldML();
 *   await ml.loadModels();
 *   const forecast = await ml.predictMultiStep(temps, hums, winds, 24);
 */

import * as ort from 'onnxruntime-web';

// ---- Types ----

export interface WBGTForecast {
  temperature: number;
  humidity: number;
  windSpeed: number;
  wbgt: number;
  riskLevel: 'Low' | 'Moderate' | 'High' | 'Very High' | 'Extreme';
  workCapacityPercent: number;
  recommendation: string;
  timestamp: Date;
}

// ---- Constants (must match Python training pipeline) ----

const REQUIRED_HISTORY = 73; // max lag (72) + 1

const WBGT_THRESHOLDS = {
  low: 26.0, moderate: 28.0, high: 30.0, veryHigh: 32.0,
};

// ---- WBGT Calculation (ISO 7243) ----

function calculateWBGT(tempC: number, humPct: number, windMs: number): number {
  const T = tempC, RH = humPct;
  // Stull (2011) psychrometric wet-bulb approximation
  const Tnwb =
    T * Math.atan(0.151977 * Math.sqrt(RH + 8.313659)) +
    Math.atan(T + RH) - Math.atan(RH - 1.676331) +
    0.00391838 * Math.pow(RH, 1.5) * Math.atan(0.023101 * RH) - 4.686035;
  const Tg = T + 2.0;  // simplified globe temp
  return Math.round((0.7 * Tnwb + 0.2 * Tg + 0.1 * T) * 10) / 10;
}

function getRiskLevel(wbgt: number): WBGTForecast['riskLevel'] {
  if (wbgt < WBGT_THRESHOLDS.low) return 'Low';
  if (wbgt < WBGT_THRESHOLDS.moderate) return 'Moderate';
  if (wbgt < WBGT_THRESHOLDS.high) return 'High';
  if (wbgt < WBGT_THRESHOLDS.veryHigh) return 'Very High';
  return 'Extreme';
}

function getRecommendation(r: WBGTForecast['riskLevel']): string {
  const recs: Record<string, string> = {
    Low:         'Safe for all outdoor work. Stay hydrated.',
    Moderate:    'Limit heavy physical work. Take 15-min breaks each hour.',
    High:        'Reduce work intensity. 30-min rest per hour. Avoid peak sun.',
    'Very High': 'Only light work permitted. 45-min rest per hour minimum.',
    Extreme:     'STOP outdoor work. Dangerously high heat stress risk.',
  };
  return recs[r];
}

function workCapacityPercent(wbgt: number): number {
  if (wbgt <= 18) return 100;
  if (wbgt >= 40) return 22;
  return Math.round(100 - (wbgt - 18) * (78 / 22));
}

// ---- Feature Engineering (must match Python exactly) ----
// 17 features in this exact order:
//   lag_1..lag_72, hour_sin, hour_cos, doy_sin, doy_cos,
//   rolling_mean_24h, rolling_mean_72h, rolling_std_24h,
//   delta_1h, delta_24h

function createFeatureVector(
  history: number[], currentHour: number, dayOfYear: number,
): Float32Array {
  const n = history.length;
  const f = new Float32Array(17);

  f[0]=history[n-1]; f[1]=history[n-2]; f[2]=history[n-3];
  f[3]=history[n-6]; f[4]=history[n-12]; f[5]=history[n-24];
  f[6]=history[n-48]; f[7]=history[n-72];

  f[8] =Math.sin(2*Math.PI*currentHour/24);
  f[9] =Math.cos(2*Math.PI*currentHour/24);
  f[10]=Math.sin(2*Math.PI*dayOfYear/365.25);
  f[11]=Math.cos(2*Math.PI*dayOfYear/365.25);

  const last24=history.slice(n-24), last72=history.slice(n-72);
  const mean24=last24.reduce((a,b)=>a+b,0)/24;
  f[12]=mean24;
  f[13]=last72.reduce((a,b)=>a+b,0)/72;
  f[14]=Math.sqrt(last24.reduce((s,v)=>s+(v-mean24)**2,0)/24);
  f[15]=history[n-1]-history[n-2];
  f[16]=history[n-1]-history[n-24];

  return f;
}

// ---- Main Class ----

export class HeatShieldML {
  private sessions: Record<string, ort.InferenceSession> = {};
  private _isLoaded = false;

  constructor(private modelBasePath = '/models') {}

  get isLoaded() { return this._isLoaded; }

  async loadModels(
    onProgress?: (loaded: number, total: number, name: string) => void,
  ): Promise<void> {
    ort.env.wasm.wasmPaths = '/';
    const names = ['temperature', 'humidity', 'windspeed'];
    for (let i = 0; i < names.length; i++) {
      onProgress?.(i, names.length, names[i]);
      this.sessions[names[i]] = await ort.InferenceSession.create(
        `${this.modelBasePath}/${names[i]}_model.onnx`,
        { executionProviders: ['wasm'] },
      );
    }
    onProgress?.(names.length, names.length, 'done');
    this._isLoaded = true;
  }

  private async predict1(name: string, features: Float32Array): Promise<number> {
    const s = this.sessions[name];
    const t = new ort.Tensor('float32', features, [1, features.length]);
    const r = await s.run({ [s.inputNames[0]]: t });
    return (r[s.outputNames[0]].data as Float32Array)[0];
  }

  async predictWBGT(
    temps: number[], hums: number[], winds: number[],
    hour: number, doy: number,
  ): Promise<WBGTForecast> {
    if (!this._isLoaded) throw new Error('Call loadModels() first');
    const [t, h, w] = await Promise.all([
      this.predict1('temperature', createFeatureVector(temps, hour, doy)),
      this.predict1('humidity',    createFeatureVector(hums, hour, doy)),
      this.predict1('windspeed',   createFeatureVector(winds, hour, doy)),
    ]);
    const ct=Math.max(-10,Math.min(50,t));
    const ch=Math.max(0,Math.min(100,h));
    const cw=Math.max(0,Math.min(30,w));
    const wbgt = calculateWBGT(ct, ch, cw);
    const risk = getRiskLevel(wbgt);
    return {
      temperature: Math.round(ct*10)/10,
      humidity: Math.round(ch*10)/10,
      windSpeed: Math.round(cw*10)/10,
      wbgt, riskLevel: risk,
      workCapacityPercent: workCapacityPercent(wbgt),
      recommendation: getRecommendation(risk),
      timestamp: new Date(),
    };
  }

  async predictMultiStep(
    temps: number[], hums: number[], winds: number[],
    steps = 24, startTime = new Date(),
  ): Promise<WBGTForecast[]> {
    if (!this._isLoaded) throw new Error('Call loadModels() first');
    const ts=[...temps], hs=[...hums], ws=[...winds];
    const forecasts: WBGTForecast[] = [];
    for (let s = 0; s < Math.min(steps, 72); s++) {
      const ft = new Date(startTime.getTime() + (s+1)*3600000);
      const hr = ft.getHours();
      const dy = Math.floor(
        (ft.getTime()-new Date(ft.getFullYear(),0,0).getTime())/86400000
      );
      const fc = await this.predictWBGT(
        ts.slice(-REQUIRED_HISTORY), hs.slice(-REQUIRED_HISTORY),
        ws.slice(-REQUIRED_HISTORY), hr, dy,
      );
      fc.timestamp = ft;
      forecasts.push(fc);
      ts.push(fc.temperature); hs.push(fc.humidity); ws.push(fc.windSpeed);
    }
    return forecasts;
  }

  async dispose() {
    for (const s of Object.values(this.sessions)) await s.release();
    this._isLoaded = false;
  }
}
'''.strip()

ts_path = os.path.join(DRIVE_OUTPUT, 'ml-inference.ts')
with open(ts_path, 'w') as f:
    f.write(ts_code)

print(f'\u2713 ml-inference.ts saved to Google Drive ({len(ts_code):,} chars)')
print(f'  Copy to: src/ml-inference.ts in your Vercel project')
print(f'  Install: npm install onnxruntime-web')